# ДЗ 6.1. Перенос импульса по кольцу

**Константин Мурусидзе, 6 курс мехмата МГУ · уровень 🟡**

Сравниваю схемы upwind и Лакса–Вендроффа для переноса с единичной скоростью
по кольцу длины 1. Сначала проверяю схемы на синусоиде, затем заменяю её
прямоугольным импульсом и исследую влияние числа Куранта.

[Условия курса, занятие 6](https://github.com/yartmihant/Students6Year2026/blob/main/README.md#занятие-6-уравнение-переноса).

## Сетка, схемы и точное решение

Для $u_t+u_x=0$ точный профиль равен $u_0((x-t)\bmod 1)$.
Один оборот занимает единицу времени. Использую $N$ узлов $x_j=j/N$,
$j=0,\ldots,N-1$, шаг $h=1/N$ и периодические индексы. Точка $x=1$
не входит в расчётную сетку.

При $C=\Delta t/h$ формулы одного шага имеют вид

$$
u_j^{n+1}=(1-C)u_j^n+C u_{j-1}^n \qquad\text{(upwind)},
$$

$$
u_j^{n+1}=\tfrac12 C(C+1)u_{j-1}^n+(1-C^2)u_j^n
+\tfrac12 C(C-1)u_{j+1}^n \qquad\text{(Лакс–Вендрофф)}.
$$

Ошибка и количество примеси: $E_1=h\sum_j|u_j-u(x_j,t)|$,
$M=h\sum_j u_j$. Сумма коэффициентов каждой схемы равна 1, а циклический
сдвиг не меняет суммы значений. Поэтому $M$ сохраняется с точностью
округления, пока значения решения не вырастают настолько, что округление существенно.

Расчёты идут с $\Delta t=Ch$ на всех сетках. Для снимка между полными
слоями делаю отдельный короткий шаг от предыдущего слоя; сам расчёт
продолжается с полным шагом. Так можно сравнить решения в общие моменты
времени, не меняя выбранное число Куранта во всей траектории.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": 0.25})
methods = ("upwind", "LW")
labels = {"upwind": "Upwind", "LW": "Лакс–Вендрофф"}
colors = {"upwind": "#b95c18", "LW": "#24609f"}
grids = (50, 100, 200)
stable_c = (0.5, 0.8, 1.0)
c_colors = {0.5: "#b95c18", 0.8: "#24609f", 1.0: "#19765b"}
snapshot_times = (1.0, 5.0)
times = np.linspace(0.0, 5.0, 251)


def grid(n):
    return np.arange(n)/n


def exact_grid(n, t, profile):
    if profile == "sine":
        return 1 + 0.5*np.sin(2*np.pi*(grid(n) - t))
    # Координата в единицах h. Округляем аргумент, чтобы попадание
    # узла точно на фронт не зависело от погрешности записи t.
    j = np.mod(np.round(np.arange(n) - n*(t % 1), 11), n)
    return ((j >= n/5) & (j < 2*n/5)).astype(float)


def step(u, cfl, method):
    left = np.roll(u, 1)
    if method == "upwind":
        return (1-cfl)*u + cfl*left
    right = np.roll(u, -1)
    return (0.5*cfl*(cfl+1)*left + (1-cfl*cfl)*u
            + 0.5*cfl*(cfl-1)*right)


def solve(n, cfl, method, profile, output_times):
    dt = cfl/n
    u = exact_grid(n, 0.0, profile)
    states = []
    layer = 0
    for t in output_times:
        target = int(np.floor(t/dt + 1e-12))
        for _ in range(layer, target):
            u = step(u, cfl, method)
        layer = target
        short_c = n*(t - layer*dt)
        sample = step(u, short_c, method) if short_c > 1e-11 else u
        states.append(sample.copy())
    return np.asarray(states)


def metrics(states, reference):
    return {
        "E1": np.mean(np.abs(states - reference), axis=1),
        "min": np.min(states, axis=1),
        "max": np.max(states, axis=1),
        "M": np.mean(states, axis=1),
    }


def index_at(t):
    return int(np.argmin(np.abs(times-t)))


def plot_periodic(ax, n, u, **kwargs):
    # Замыкание кривой для рисунка; дополнительный узел не участвует в счёте.
    return ax.plot(np.r_[grid(n), 1.0], np.r_[u, u[0]], **kwargs)


def table(headers, rows):
    def row_text(row):
        return "| " + " | ".join(str(item).replace("|", r"\|") for item in row) + " |"
    lines = [row_text(headers),
             "| " + " | ".join(["---"]*len(headers)) + " |"]
    lines.extend(row_text(row) for row in rows)
    display(Markdown("\n".join(lines)))

## Гладкий профиль · 🟢

Начальный профиль $1+\tfrac12\sin(2\pi x)$ переношу с $C=0.8$.
Профили показываю на $N=50$, где отличие от точного решения заметнее.
Ошибки считаю для трёх сеток после одного и пяти оборотов.

In [ ]:
sine = {}
sine_stats = {}
rows = []
for n in grids:
    exact = np.array([exact_grid(n, t, "sine") for t in times])
    for method in methods:
        key = (n, method)
        sine[key] = solve(n, 0.8, method, "sine", times)
        stat = sine_stats[key] = metrics(sine[key], exact)
        drift = np.max(np.abs(stat["M"] - stat["M"][0]))
        rows.append((n, labels[method], f'{stat["E1"][index_at(1)]:.3e}',
                     f'{stat["E1"][index_at(5)]:.3e}', f"{drift:.2e}"))
table(("N", "Схема", "E₁ при t=1", "E₁ при t=5", "max |M(t)−M(0)|"), rows)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.7), sharey=True, layout="constrained")
x_fine = np.linspace(0, 1, 501)
for ax, t in zip(axes, snapshot_times):
    ax.plot(x_fine, 1+0.5*np.sin(2*np.pi*(x_fine-t)), "k--", lw=1.6, label="Точное")
    for method in methods:
        plot_periodic(ax, 50, sine[50, method][index_at(t)], color=colors[method],
                      label=labels[method])
    ax.set(xlabel="x", title=f"t = {t:g}, N = 50", xlim=(0, 1))
axes[0].set_ylabel("u")
axes[1].legend(loc="lower left")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6), layout="constrained")
for ax, t in zip(axes[:2], snapshot_times):
    for method in methods:
        ax.loglog(1/np.array(grids), [sine_stats[n, method]["E1"][index_at(t)] for n in grids],
                  "o-", color=colors[method], label=labels[method])
    ax.set(xlabel="h", ylabel="$E_1$", title=f"Ошибка при t = {t:g}")
    ax.legend()
for method in methods:
    mass = sine_stats[50, method]["M"]
    axes[2].plot(times, mass-mass[0], color=colors[method], label=labels[method])
axes[2].set(xlabel="t", ylabel="$M(t)-M(0)$", title="Количество примеси, N = 50")
axes[2].ticklabel_format(axis="y", style="sci", scilimits=(0, 0), useOffset=False)
axes[2].legend()
plt.show()

Обе схемы сохраняют $M(0)=1$, но ошибки профиля различаются. В частности,
upwind снижает амплитуду синусоиды. После одного оборота удвоение $N$
уменьшает $E_1$ примерно вдвое у upwind и вчетверо у Лакса–Вендроффа.
Сохранение общего количества не гарантирует правильного распределения примеси.

## Прямоугольный импульс · 🟡

Теперь $u_0=1$ на $[0.2,0.4)$ и $u_0=0$ вне этого интервала.
Для $C=0.5,\ 0.8,\ 1$ вычисляю $E_1$, минимум и максимум на трёх
сетках. Значения вне диапазона $[0,1]$ сохраняю как есть.
На рисунке оба фронта при $t=1$ показаны в одинаковых локальных
координатах и с одинаковыми пределами по вертикали.

In [ ]:
pulse = {}
pulse_stats = {}
for n in grids:
    exact = np.array([exact_grid(n, t, "pulse") for t in times])
    for cfl in stable_c:
        for method in methods:
            key = (n, cfl, method)
            pulse[key] = solve(n, cfl, method, "pulse", times)
            pulse_stats[key] = metrics(pulse[key], exact)

rows = []
for cfl in stable_c:
    for method in methods:
        for n in grids:
            stat = pulse_stats[n, cfl, method]
            values = []
            for t in snapshot_times:
                i = index_at(t)
                values += [f'{stat["E1"][i]:.3e}',
                           f'{stat["min"][i]:.3f} / {stat["max"][i]:.3f}']
            rows.append((f"{cfl:g}", labels[method], n, *values))
table(("C", "Схема", "N", "E₁, t=1", "min / max, t=1",
       "E₁, t=5", "min / max, t=5"), rows)

fig, axes = plt.subplots(2, 3, figsize=(12, 6.5), sharex=True, sharey=True,
                         layout="constrained")
n = 100
i = index_at(1)
u_at_one = np.array([pulse[n, c, method][i] for c in stable_c for method in methods])
limits = (u_at_one.min()-0.07, u_at_one.max()+0.07)
for row, (front, rising) in enumerate([(0.2, True), (0.4, False)]):
    for col, cfl in enumerate(stable_c):
        ax = axes[row, col]
        ax.stairs([0, 1] if rising else [1, 0], [-0.13, 0, 0.13],
                  baseline=None, color="black", ls="--", lw=1.5, label="Точное")
        for method in methods:
            ax.step(grid(n)-front, pulse[n, cfl, method][i], where="post",
                    color=colors[method], marker=".", ms=3, label=labels[method])
        ax.set(xlim=(-0.13, 0.13), ylim=limits,
               title=f"{'Передний' if not rising else 'Задний'} фронт, C = {cfl:g}")
        if row == 1:
            ax.set_xlabel("$x-x_{\\mathrm{ф}}$")
    axes[row, 0].set_ylabel("u")
axes[0, -1].legend(loc="center right", fontsize=8)
fig.suptitle("Фронты после одного оборота, N = 100")
plt.show()

### Перенос импульса во времени

В анимации $N=100$, $C=0.8$. Чёрная ступенька движется с точной скоростью,
цветные кривые — численные профили в тот же момент. Изображение
сохраняется в выводе ячейки как GIF.

In [ ]:
import base64
from pathlib import Path
from tempfile import TemporaryDirectory
from matplotlib.animation import FuncAnimation, PillowWriter

frame_times = np.linspace(0.0, 5.0, 101)
n_anim = 100
c_anim = 0.8
frames = {method: solve(n_anim, c_anim, method, "pulse", frame_times)
          for method in methods}


def pulse_segments(t):
    edges = np.unique([0.0, (0.2+t) % 1, (0.4+t) % 1, 1.0])
    middle = (edges[1:]+edges[:-1])/2
    phase = (middle-t) % 1
    heights = ((phase >= 0.2) & (phase < 0.4)).astype(float)
    return np.repeat(edges, 2)[1:-1], np.repeat(heights, 2)


fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
exact_line, = ax.plot([], [], "k--", lw=1.5, label="Точное")
lines = {method: ax.plot([], [], color=colors[method], lw=1.7, label=labels[method])[0]
         for method in methods}
values = np.array(list(frames.values()))
ax.set(xlim=(0, 1), ylim=(min(0, values.min())-0.08, max(1, values.max())+0.08),
       xlabel="x", ylabel="u")
title = ax.set_title("")
ax.legend(loc="upper right")
x_closed = np.r_[grid(n_anim), 1.0]


def draw_frame(i):
    t = frame_times[i]
    exact_line.set_data(*pulse_segments(t))
    for method, line in lines.items():
        u = frames[method][i]
        line.set_data(x_closed, np.r_[u, u[0]])
    title.set_text(f"Перенос по кольцу · C = 0.8 · t = {t:.2f}")


animation = FuncAnimation(fig, draw_frame, frames=len(frame_times),
                          interval=100, cache_frame_data=False)
with TemporaryDirectory() as directory:
    path = Path(directory)/"advection.gif"
    animation.save(path, writer=PillowWriter(fps=10), dpi=85)
    gif = base64.b64encode(path.read_bytes()).decode("ascii")
plt.close(fig)
display({"image/gif": gif, "text/plain": "Анимация переноса, 0 ≤ t ≤ 5"}, raw=True)

### Минимум, максимум и ошибка

Сначала сравниваю устойчивые расчёты на $N=100$ до $t=5$. Для этих
графиков беру моменты $t=0,0.02,\ldots,5$; при $C=1$ они приходятся
на полные временные слои. Случай $C=1.1$ показываю отдельно до $t=1$,
поскольку рост выбросов меняет масштаб на много порядков.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, layout="constrained")
for row, method in enumerate(methods):
    for col, (quantity, name) in enumerate([("min", "min u"), ("max", "max u"), ("E1", "$E_1$")]):
        ax = axes[row, col]
        for cfl in stable_c:
            ax.plot(times, pulse_stats[100, cfl, method][quantity],
                    color=c_colors[cfl], label=f"C={cfl:g}")
        ax.set(title=f"{labels[method]}: {name}", ylabel=name, xlim=(0, 5))
        if row == 1:
            ax.set_xlabel("t")
axes[0, -1].legend()
plt.show()

t_unstable = np.linspace(0, 1, 101)
n = 100
reference = np.array([exact_grid(n, t, "pulse") for t in t_unstable])
unstable = {}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), layout="constrained")
rows = []
for method in methods:
    states = solve(n, 1.1, method, "pulse", t_unstable)
    stat = unstable[method] = metrics(states, reference)
    for ax, quantity, name in zip(axes, ["min", "max", "E1"], ["min u", "max u", "$E_1$"]):
        ax.plot(t_unstable, stat[quantity], color=colors[method], label=labels[method])
        ax.set(xlabel="t", ylabel=name, title=name)
        ax.set_yscale("symlog", linthresh=1)
    rows.append((labels[method], *(f'{stat[q][-1]:.3e}' for q in ("min", "max", "E1"))))
fig.suptitle("Неустойчивые расчёты: C = 1.1, N = 100")
axes[-1].legend()
plt.show()
table(("Схема", "min u при t=1", "max u при t=1", "E₁ при t=1"), rows)

## Что видно из сравнения

При $0<C<1$ upwind усредняет соседние значения с неотрицательными весами,
поэтому сохраняет диапазон $[0,1]$, но размывает фронты. У Лакса–Вендроффа
вес правого соседа $C(C-1)/2$ отрицателен. Более высокая точность на гладком
профиле не обеспечивает монотонности на ступеньке: рядом с фронтами
появляются значения ниже нуля и выше единицы. При сгущении сетки ошибка
$E_1$ уменьшается; величина отдельных выбросов около разрыва не обязана
уменьшаться вместе с интегральной ошибкой. Например, при $C=0.8$ и $t=1$
переход от $N=100$ к $N=200$ снижает $E_1$ Лакса–Вендроффа с $0.052$
до $0.035$, но минимум меняется с $-0.175$ до $-0.195$.

При $C=1$ обе формулы сводятся к $u_j^{n+1}=u_{j-1}^n$. За шаг
точный профиль тоже проходит ровно один узел, поэтому на полных слоях
обе схемы воспроизводят его значения. После одного и пяти оборотов
в таблице получаются $E_1=0$, минимум 0 и максимум 1.

При $C=1.1$ коротковолновые возмущения усиливаются. Например, для
сеточной моды $(-1)^j$ множители за полный шаг равны $1-2C=-1.2$
и $1-2C^2=-1.42$. Их модули больше единицы; соседние по частоте моды
также растут. На графиках это проявляется быстрым ростом обоих выбросов
и ошибки, особенно для Лакса–Вендроффа.